# Meta Seal quickstart (wmkit)

Load Pixel Seal, watermark a Kodak image, read the message back, measure robustness, then probe the
white-box adversarial robustness of the public extractor (`pgd_remove`, `pgd_forge`).

Run from the repository root with `WMKIT_DATA_DIR` and `WMKIT_WEIGHTS_DIR` set; the first `load_seal`
downloads the checkpoint (`scripts/fetch_weights.py`), Kodak comes from
`scripts/fetch_data.py --dataset kodak`.

In [ ]:
import io

import numpy as np
import torch
from PIL import Image

from wmkit.adversarial import pgd_forge, pgd_remove
from wmkit.config import data_dir, get_device
from wmkit.seal import load_seal

SEED = 0
device = str(get_device())
print("device:", device)

## 1. Load the model and an image

In [ ]:
seal = load_seal(card="pixelseal", device=device)
print(seal.name, seal.n_bits, "bits, scaling_w =", seal.scaling_w)

path = sorted((data_dir() / "kodak").glob("*.png"))[0]
image = torch.from_numpy(np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0)
image = image.permute(2, 0, 1).unsqueeze(0)  # [1, 3, H, W] in [0, 1]
print(path.name, tuple(image.shape))

## 2. Watermark and read the message

In [ ]:
def bit_accuracy(soft, message):
    return float(((soft > 0.5).float() == message).float().mean())


def psnr(a, b):
    return float(10 * torch.log10(torch.tensor(1.0 / float(((a - b) ** 2).mean()))))


def to_8_bit(x):
    return (x.clamp(0, 1) * 255).round() / 255


generator = torch.Generator().manual_seed(SEED)
message = (torch.rand(1, seal.n_bits, generator=generator) > 0.5).float()

marked = to_8_bit(seal.embed(image, message))
print(f"PSNR {psnr(image, marked):.1f} dB, bit accuracy {bit_accuracy(seal.extract(marked), message):.3f}")
print(f"unmarked image, bit accuracy {bit_accuracy(seal.extract(image), message):.3f}  (chance = 0.5)")

## 3. Robustness to a JPEG round trip

In [ ]:
def jpeg(x, quality):
    buffer = io.BytesIO()
    array = (x[0].permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
    Image.fromarray(array).save(buffer, format="JPEG", quality=quality)
    decoded = np.asarray(Image.open(io.BytesIO(buffer.getvalue())).convert("RGB"), dtype=np.float32)
    return torch.from_numpy(decoded / 255.0).permute(2, 0, 1).unsqueeze(0)


for quality in (90, 70, 50):
    print(f"JPEG {quality}: bit accuracy {bit_accuracy(seal.extract(jpeg(marked, quality)), message):.3f}")

## 4. White-box probes (the Pixel Seal weights are public)

`pgd_remove` pushes the extractor's logits to 0 (reads random bits); `pgd_forge` makes it read a chosen
message on a clean image. Budget: L-inf 4/255, 50 steps (about 2 s per image on Apple MPS).

In [ ]:
removed = to_8_bit(pgd_remove(seal.extract_logits, marked, eps=4 / 255, step=1 / 255, steps=50))
print(f"remove: bit accuracy {bit_accuracy(seal.extract(removed), message):.3f}, "
      f"PSNR vs watermarked {psnr(marked, removed):.1f} dB")

target = (torch.rand(1, seal.n_bits, generator=generator) > 0.5).float()
forged = to_8_bit(pgd_forge(seal.extract_logits, image, target, eps=4 / 255, step=1 / 255, steps=50))
print(f"forge: bit accuracy vs target {bit_accuracy(seal.extract(forged), target):.3f}, "
      f"PSNR vs clean {psnr(image, forged):.1f} dB")